# Taiwan ENSEMBLE: pretrained continuation and expanding-window validation

Data: 2017-01-03 through 2026-11-23; 4,461,650 rows. Exact supplied split archives are verified by SHA-256. Original GRU + RNN + LSTM NoReLU weights and source scaler are preserved.

40 expanding windows: initial 90 calendar days, inner validation 30 days, outer test 90 days. Every label is purged by its actual completion date. Training starts with 2017 history; the first outer window starts 2017-04-03.

Two summaries: all outer validation windows, and signal dates from 2021-07-01 onward. The pretrained Japanese checkpoint used data through June 2021, so earlier windows are retrospective transfer analysis rather than a point-in-time deployable backtest.

Portfolio: top 3 long / bottom 3 short, linear rank weights 2:1.5:1, total gross exposure 1. Report gross returns, a stated transaction-cost scenario and extreme-return sensitivity. Actual borrow availability, locked-limit fills, delisted-history completeness and cash dividends are not validated by the supplied data.

Uses available GPUs with one shared model and global batch size 4096. Strict weight loading, dual-GPU inference parity and gradient-update smoke checks run before full training. Source files, audit, fold predictions, weights and checkpoint are preserved as notebook output. A 10-hour budget saves a resumable checkpoint if necessary; a partial run is never reported as complete.


In [ ]:
# Taiwan ENSEMBLE: exact pretrained weights, verified source data, dual-GPU smoke test
from pathlib import Path
import base64, hashlib, json, shutil, subprocess, sys, urllib.request, zlib
import torch
ROOT = Path('/kaggle/working/tw_ensemble_20261125')
ROOT.mkdir(exist_ok=True)
RESULTS = Path('/kaggle/working/TW_ENSEMBLE_results')
RESULTS.mkdir(exist_ok=True)
assert torch.cuda.is_available(), 'Select GPU T4 x2 in Settings first'
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True), flush=True)
print('PyTorch', torch.__version__, 'CUDA', torch.version.cuda, 'GPU count', torch.cuda.device_count(), flush=True)

# Embedded reviewed Python source; decoded files are preserved in notebook outputs.
payload = zlib.decompress(base64.b64decode('eNrtfYty20aW6K9gncoAtEmIlGwnpo3UVWQ58YwfKkueyV6ahYIIkMQIBBAAlCi7vN++59Hd6MaDkrMzdXf3jiuxSaD79Onu0+fdh18ebLIwSko3v30wtR7EmzwrKqvKisX6U7ossg1/tsSLNP2Ufufd+cdioNbdLb/7lC6SoCytk3fv/DfnF2+dNHXfZuE2iQbTT6kFf8Joafl+nMaV7zv8CP+UUbIc1l/T7cZfRkG1LaJyasVppb1bpKmfbSt/sQ7SFPCi997Tx1qTq6iAN34Zf4747UR7uY7DMErppTcZj/mFRE/i4mogLE8HWLfTemzzqHAGrprYoAENcAYosBgnWXo9CbWJ4584VZPx9JkPzWb6pL3mKjTaagh72udGq7Iq4jAyVgf/5EEYxunKG9ePB2YL/U9jqmGR5YDYJOQJv5Rfndwbu4cD6zvrbXAVWdU6sm7WWRJZYbyJ0jLOUus6SLa42uNukAbABrgS8AVYHRAasJKy2jAgos/mRuSwpLRorfW1zKY6FWmfm81wO5PgNipK76j57jKoFmt/GRdl5V0U21ZfMW2cZ+dGNGcWp1FQiLnRZwcIS9KSiSTOTL2ZCJj1AV1mxU1QhA6dSms3ZbbhXsA+ZcXAGv1kPNCOzne8hFP87ND8hlYF+zu05GgDvXER/b6NiyicqsaK9Kmb1ngHE9u5VRGkZZ6VkTMZWoeN1/KkObvORfoOuZKVRJVVZhsglWgBk7Dk/lqrDCZljXFyYbQooqAE0ozyKIVVW8RRaQGBLqKiCuJUdhIwOrBQhwBwsbqRQfJjErBqErBGMPw9Fu5nOA1pdmPFpWpdE+q3rRygCTs2tJy1nwIUPx3ISeBRMdbyO+tCHl3g8pVFhD3kzzioBVwwzkIB0sSAn82mQ2sECEznAExiDkx4YDkEAOZTRJvsOgqNYU+CFJY/uaUjAZ/EQR9a8dLa4j7J5R5yM4AB21SK5b2Jq7UVJAlsLnUr+zfMmG4OlBkvKuRMnn7EeuiriGCfUq2X3kgKxtN356dvf35z+s8QjNoJF1Lv8MdGV2ZFbZkYRtfxAnqBQIC5fnqwyLefHnRIxj1yDto/kH26uBuR9sn2sxUGVWAlWRBGhbUBepLbFFREWdSjFgr9AzTY36rYMu/75cPHfra+R7z2cfRehl5/HN7B2O/DvgupInx49+5/Iv73Faz/fWcA45URT+EcBFOUVnGQNCbyrbL16HAwvA+Eo8OWUL4fzv2qUbfYuYAjlgdwjK0yjxbxMl4AZ7y1gvIqClHu0xnE0w1z/xyYbAxfpwEKpTwogg00YyTwWejTs6iCzXAGUxwqybIr4rsVjSnfNihjCQf7JopX6+rTA4SIwKZdeuZ31uulFVcoI0T7lkoKa4Cou7vgOo4Kf5vGgPLGd2jwgYLzEbb5N2qCIh8EY5htYA3ErFHjJzlhwo8SQvUyDsr7I4qtLecc9I1qDRqqlcRCfl4ihFsv2FmPrMtB/0w+R0VWNiZQj3MOyoxcYIG9EHGky7TB6pRAejOgWWxTUm5YBjToq8ocfu7xPy2q6lUaB4Za+BoZgFWugfKm1gyMNesQ1IDHcyDcGZxiK1vymZWan1/mAagj4pU8FXMd5HvSKCRM/Y1ljfAwaQOBJKShSqJe/ls7tPNm77VlQW/oiACoM/auFZ2yivJhjXEfKDxsQi3Qpz6ZG+qXD6ILYPgswaQwM7QMbAUCAluxnJAio9UK2TApc/zB79fmTC3IofFno8kcKJJGUZ+xJ34ZWAfW0R/WnYi7fpvqVKtN6E/4f6g6dbsczDYtn8OT/8W61/86p8Y3qJB3IPk/Qo387z2HP+yj+Zc6+S918l/q5P8H6uQrGAi9ecQMrFmn89Av6dQtdJUMdRZ2Xdb+sB0oZH/AmfiBdZbLYHGFq5MCZcBykZr3reP9SwfcpwNKTfDB0HoAqwhsrroRAS5WS/4SrFZJNIKe9BoYj1QV0Zl7EcQ3QWpFOzAoUAEY3cRwQG+QsOOQONIB7mEVlZWLw/wal7BjyMcApRIYK6hHWwwvoKexGIEalicxjJFDm6h0mQNmRbXMkjjDQxrAkMUqqkY8o09pXmS72ymejTCDo5RmlQWDBZdJXK6ty6woAJvgOoiT4DJO4uoWCDlO6i/IRLOK0EF4JWIp1DEK6fn+cotk7/syrgfSEDrg1EqckXxarODcl5F6sMjyW/VlHZTrJL5U3/9eZnXXrFQfmb2pr+Vt/QoPXf2lCBYRLqx68jnOYV6RwDoPKhxPonwGXzVcQRLnKEWsNFfPcP/gCfyXh/UwWmxThCnFG0kDCPXD+/cXQIM4iAPLBVj4/sCF7c2S6wh0algXEM+f0lenxxcfP5yeo7Fnv8+j1B5a9q8gHPDfN9kN/nOSwCHGD3/Nku2GPp3uQPZVUfgyvgbNMw1t4Dinv52dnlycvvT/dvr6l18v/F+Pz38FsPb4h2fPlj9Ey8Wzox8mh88eP356+Pjx5IfwcPzkcfDs8DA6evrj42fhk8WTH8LlD08eh2EQPXny7PLHy/DpcjmO7E/pyft3r17/AsDwoDhA2FGBGnvl2Rd/U1aSTwxnCWLrcHz4dDI5fOJf0TnxryeAMwpT3B3vsKknNwKkYOPoKhJwLi0sBaZ0FIWeHKEFiY8r0HoByB2OJz+MxpPR+AjGx+NmvnjML9Cy9sPgtvSejVvwYlDzCr8+udzwaDwEY2bnR3m2WJfekzEqFFWMrN+btIFsACUglCrwJtHoCSxFAVMZjyfCnud5Px4/ewrIrLfLZRJ5r4KkjFqAhLjxS+AVUeHZQISfYd2AeaziNEj8P5/9BhMCtrSI0ZqCBmB5VUeHfrU8OoQ3m21Sxf4q33r2S7DLzkAoJwnYNDdrgCIYQgIE1hxXcRvPrrL8yL/MqirbHPkcnfABfuZPgJNnZelP2t3LbFssIh9YRHQJVODZ66rKy+nBwc3Njcs04i6yzcECjtNBECMak/Hk4O/5boTsewM4jUDaRcl2hMOPVkH+bNw7zDVQDU4ellc8YaJAXwpsXxXh/h8iVYzGP/ZDqbe81fFpZ0eYwSYucWg/SyP/JrjlXX4MZIrCKPGrYEePjtqDJnGeB6vI6DnGfmtYep9Ztg9cdhsk+O5ogEzmU4pqCygAh0+eOsjepMqyhqMq2KsrXgsJSJEp4krU3s2A5Th2cWkPkM0tp6auTEpeDDqvkwSbyzCYWiD+oyB0JuPDx9ZDC/8ZABXbtq4sEQbuNsclc6RuKITt2l1HuzBewWF0tDkskihIfZQAjtK7QFsFKk7hyMKxQh0GmY8+jgD5Bcxv5wo0dg3I9YDwvxpa1ziFnQuT2IBa/7UHtFNt8wTMAhCQ1aBjjFkbNgOe9wBMcxdkR1EEtx3Q9NmCgoqDOoNBP6RVBEwoXtwFCefYD4dYAexyGpI+AGDjcok6d+TsOiC/A0rsWyykH2AzoXsBUgCeb/KuNcNt2Zm7v6t3HHH2ywCEYU7QSOuXUPCRlJ5Ep/wY5pfDY3ziIiH7aMFQA7LgQG203WqT21pz96aAGfpVtKscHNINQc6Xjr6dNPAQdjNEiQYSBngOMtmgXMSxYMRopMABTIOUH8hVzko4D3kCioeDo5EYWGt0TcrCP3uapO3TGCJGjQDuiSDIClRFHHTT+YziYh0trvIM3ZWo2ytkC3ji2GcfTs+OP7x+94v/8vji2Ea62pZr8lHUlIdbryAOXFBQSpyIY7ugkhm8gviR0NPc/xvnr+BfrSsypc8N1hKUoBWDckeLwWfH8lB9Ai16ceWTlgwaF8hDd1Fe2/Oe3qgPwMDQGTTomtpNxAR/7IbcYpnqAAQ3uH8h8UofmjpLsTZkg/evDmwFGJCVsUImMNFC6y8Bl1FvJ8Sg2QFazFAJiOw5twQRjgwbVWpHeymai2VDzgEv4RSBSbKABqEjGoJSeh4ttnDa4qg8AUEOXd0gvXXq8WAc7FuiRGMjHzo3OgEYMS4qy1Hl47HcOagB6jQm0EFw2NwFDYupQGdKDqutM1tTCpsTwqm6x+Hft2BtwvF/FVCKzk/WGJBPEon8SqAOGs42v7x12ljjpARr4C6ooaEEQMdWTnN2FLbbNIZNVKw6xJgTNyPjDxvDwkoIw3qasEdkp6iuoK+B+gbNoP+K25TreFk5o8NGi6q4bbWRaVjsVEMKsUthSuKhiHawGsmtRfbHrHp0OAdTXH6Z4JfjV/x4ZE1cyehBDcq3gD+NRq1rlKBL49mEnzV3QOsCsMWKAmBQWEEZFLvBmLpAk2kAu/8nbXD5kHtuKOLgscxb4PBO3X+odYPPII6Flh6Ijz+aFEPAZjUycyYUIFyxcsBWyZWFHql4CfYQMAk0FoNtGOO60Drrm7MAqxnoqYqvcWaO008Iart1SkCyJxJ6ZLh4Wn/+ZN0DMpBJD+zJQOccb7D9KRqegLJCq90A4BlN4Huz0Uk9fa2ptihah7bdK2ih+cJFhwZQwNgVWAuvIrWeSat7Xs81rG7zyIMFIj0JY6borNB5jjS+yDLAjdJVKAbPTCPYxXWKY9c2cGPMTJs+ntfMRvQzW2ADD1qoo3qGfLG4Zi8sGyvsh4Czuwi2ZZBYgTpNMIn0GuMAQIwOyC9MP4TpFlk4EAcWvm6TQNBex0EUzeVRYhie1k0xRdwGky8abFEc6Qn7mlgAeBO3IV+0JaWhBBsmtVVMoMmczdU88BhHjYrxFQr4ebM9NH/Ybi54xpYZRlCSFu/UWhEoGmR+2/OZvQE10p6DadKgHgGkDO8DpKzCPTAU4TqCKoAjbrbINkvYRDBNoZcjex0dmovyHwbR4mEZu+R33wJDBJMSSSa10ghsZusSXfAWOi6TeBUjm2W/pSAUdKlKgSp9OVK5isOdmGgRpKvIAa6C5CCJ9jJaxSk32ACRb7YbB3uMAOYj9EyPZTtUcApgnwJa4c/GZAABtZXQyZzNYC7lQigPdoP+WpvKE6pPMKEBxytFKUMHTwDzGOqMMJ/3HGVoTrxxRCxStMW+DK63Uz3NGY4PS4ASVHtaQ9LO/XmUROza3paUG0wBHwwPkZfCukqzm1SG1ct4BVrqCBU6iwSeqzHRc3p5KnaZWKixMH9irYi9jdoa0snTuTeIktQAwzzAAA/Q9ENdy919oorWCZs2BYT+GKSPFPOSiaTZBmagqCHJFjNNUQguS5qEe9h20dV/evVZ3RPbZJO2Ev/kt1XCcd5ADFcTtXGMeB5YdkVdfPXaz67JfZovKrIw2CrdGYolqRHSFYtH2JOnDV2ZYKeUXsdJSKXOOUQXKUYlSNvnzlINGHQtC8Wg2f1lqtucht18E+zwjbIPfDadeKzOZRf+NrJP2FflCZdVbbIoTx7HTWW7Lo931xDkloMZZ+tk4dOSoSnbZCcuspjBYB8A3i/urhFWXCIR7ukfCe3ED4V64ucZrM+tZx8niRTixI7FSM9JSAojNCoytDSll5dYEwV2XBBrlgTJIZuRcgwF8ca1u5ARRIecA0S4Z9fq/UGt3B80KZwaYCI9Rd5xUxHhzgGAhuNlDLMVI204aY1Wrdab5XK1m8el7DHegz/QoKftgSBJNeTOeElU2Q8LOIN29OhMqDN5r35PqB9RVRfDebKPOCQV1lqbZ7//9c0JMAilsUkVbUl78dxi3mw97GthOYpaBHXViuCgc9NYYvgl2M1JVFO4yczVniXI+luNDYGwZ8bAi9BJHwnyl/FNlr1I6Zek4YYY1I8C0FTC4PY52X9JVEVpVJaYohmS8wdarSl+eosm6zaV1NQ5ySTexCJM6c3sk6zIswI40yhgwWqMgBpntKuKCG+ZoOfHwlsi8LyItGG6j5jxx4ZDuq5DvOTRV6FWwmgEguoKDM4to4FHn4KwPEMMBC+Ccj0CDe+GrMf7DPpBRpJlqBkpICSzFJCAUdSyI8TnVpoxz8CVTZCQoh3KXDK34ciTnpEuIlc5UGoHrhBmxK5JOrn4zh6yqDI9wEAmQ3UnRjohl1kSlsiQHCVRpAeMnDZWn19HhfQUVrDMebN5DRR0LfGGwnEOBfMmCsNym+BYMyG1b9ZxEgkMXhBkzcPGPhfkOtygC7LEU0UYAU0S0vlA95bjsG6Q48Uvh6Q6rgdLdXo3IC15745r0SoOcTJSoz1IdUY2EcG9A2lRVPpbxFVRpEW7RbItgXQ8+DbQMx3FHtYOAkELNDstkAX0SA4/THNwiFJwJSQpXGOWDf5PXhJjh7HZ7Gougj6gNHXNwbGbq4S6mkZD8psxGVvOhJyIbJcsgdemWYqC2VFePSCR63IglWepAqqnLf5YG2r7gIJxokEV41Sd41R7xyEWsH+gqj1QJEGacsA02pFSaXXYTMevMKv6Cw7d9tvGqJ1Tt3mtz5OghoFVAwA013TObijUqB8Gjt8LRMbCUszsAyVtEvJciEuB2MZkOj1mprWDBkz+3Myg7RoGN9FCnfkW6Q1sIfLaSOqWwcyrqXXthlEFYg9USmiMfydZCqq7EdKkzi4B8olpDLQYpxyr2Ka+uDLAow2t3dC6JbMCD5megAASJ69AGgFdeOgtkYjRTFAGeXUDFLW4IHXYhNGhto7sIUNQy2UJSrPKSxM+gglwp7Y/gTLVRMLezjVS9jglSUKhvEKHbI3O1qQ6i9BYhMluYF5HKYqzsEawEe5mRhWjlEIMx0MiX7FWA32xmuFuZBnsBBGtZwRrKmVD3bMRjtpd4l3ZmeivPFXQRSzbDL/j9dVGR8xas0ShBGd3CSQI5l30Oaq9+krpycp68dPUXW7TBasB7np7CTIAGzgID0hDYoL+KM5YcdFBHrJq4rHDa9DKgEVi4AGUpY9QB53BsbiMrFforYItOENn2GlRgOwHPSnlvoRyxyhy3zqgKtIkwqDtdnDf0XsBC6j7cZtr4yLlUeJnRwO1atsqTko4iXHOtJRSXi6TvZ44DATsDvbhh8pAcyQm7Uce4yNPP6j3SH9iR0w9irx9DvUTcX9QwXRqNWO8xCFqDtBx/OVW8YwiYFzOP/j8Kkem0q/qI5qKLfvHncg4LL/9NDKKUhcTZ2sGoDoO5tw4coJTm9E5pVGiCzNL0R+SBnWwoNfzLdQ+3cndozIB1YFV4QOEGFUmTtdUIsV0Pou30tHM+SB1y2JmoDCoM1vwcJM22pHkQYRFeuW44fMuq9ApYLAwW2JQA+AwDLCHJxSutsYyBBGkPppe2ZZ06sssS0AxgXa4rrQGMsj9u/ILI1tixzBFJyYubAzgjMMY4GigWky1sGachhbyNdCA4+TWK1z8gicZ7++gUe1zN3hDNm2HhsyZWUBQIfqmCrm8joRF3no4z7guvxeVc/jkcEBrAk9/ssY1mh2w86xkLABhf1mwrSrgOyIcwqN0YSZXAjATs+Ce0e+UaU1+795F64IXrAof0DfBPHxo4ZwEDyoGg28HjPmUIehsYXYjccR4AegTsO0HWuzADRbC3xHBKx7pG4eKtb1mvxGNLh+hAqnxT2lBO3WmuDpjnBd9Fi+u6DpFaV1GwLciynZFpQn98ew0dK237NvDXEDyBnKkswQmJQwT0DTR1UFJ4Qj8dIde1LjC+hMjeMGu/FGVjegDysOUr4Mut3gHhp6GB0WEWSsAH8M6twIiwjvJwAYo8aYd0FNc3WIFD8ANkM6QEwMDhs9OmllIJ+jXstKoQilN6VFgEIXki1gjLxa3YLA0CYyKz4FNsFZhva6kgkhp8OS7R49DzF6qLHX1xROBRaLyjPgTHEwYcEGiYij+FyKDGUASp2WOOU2HLsrboXU0qN/fALXc8EkVgVt0pcDqwjKhnEactZ2skzqE359ilqQvaOxuW4rEAwJBkQX+1BGh4BdmdoImaAqaGQN0zxQiLmzmFSr7WVE44q2AwW8UD+WXyEgPmcwx2TDQbN54YbgTcF4eLwBC8uOFBzgMVZyPmCAhPRCJAMwXxTi6KS8w4MZgYz1tSF3cur6xpXdNptRhwRvKUY5uQAmtgKr8p/4Cr0xQtMAeNJQkkKBAi/qtJpwNacG84mZiUb2yXblJwNzLRUTXM7yZQAj3XI/1gdXFiT84irvGvNcj3J86nkjyDv15dbMKmAg2+8PIcF0mxqU1GKI0s/9GURDsR4PXDwAN9wmImZuhNeIPza7nMQ8oOvJXvCWArxE3emFrV2qAtZXsUWMVxplhUwEBIMUrTM3mPK2mmi23XEp1AtY4GVqAn183T5ShAiiU0H3LwF8GnGIiB9MaSp4iSZJ6Gr63hYykaYRK7jf0aTNZ4Vd2JIzRfafIWH7odphJJu8LH7IvDhbFCIx1qMOS+6IEdRBKQdYgGgDvDEdZ6uzZJDflLhEzsRV8uXz+Oih9FUMr0KDSsnMYDdtkEhJkgz2gnmmgqoLLujaqUjCJqkvVh6l8fx+MyAghgesqQDQaXW4xF451DOowk0OBMjXnhQN9EpU1+Gi2eAEt4GCB+QMKp/F00GRYmPPfM86LO8f5qXOcn7rGQabrr6OEUhq0M4RuwZFlPKiK24EbVi7x6WZ2hFoeviAmMd+3DMrD3b6mYON7hZrZ8cA6evqkzd/LivNccH8e0fKhDazc/e0LFjbaYwqHxh0Kuy9vQ8DW0Re3Mwgez6LRl+62qCWRk/k/YHM0KDxayJsP1MWjv4c0OR+ZfFDEmUdTHaJ25ZsveJgRr0W/Jx45sI8ZIHznRqjLN4APvmkdrsEepz56NHI47BT59DXl0GtOFc0taIxG3QgvYgG/2gM3+h1oQMbr1dUiybMYtqnl9NswNdNiE4tSDkxCV0p7lyYCm2Ka0SDM8P7TK+TvZAcOhpqA01TRtnAbmp3jhWYo8F2DIsLrUhgVa+o8et4/EpPIwoSW7irJLh0bRY3/0Ne1U5USPjDscQLRMsdljh4pRZrEbmaP5+xLzkkVRkh75LjmJ0fAfyD1m+yqoa7hxwucvmFX6W0bWTKyHRtovYkxtbQ3u7fkWYkg3NVnEwpvVhGVfHNuRTcVxJovmjAJE19o070YlaDL01Yw4qhCiCkIshBWgfDisAGJNxnoVOO6tjgEPmycfnzUe4rx5d6jaBs44PAYgjD9SjwPI8Bm4PxVZjOTpsetabbiowwwGTE7cZcOL2ja0jxCCL4kXwmEyOMeMDBURU07TDJ63mmR0ZBkkeEu43j477dhvG/tsMGdKycTEr8BfcnPcrY+SIfFaNC29GxJXl3aHbAArLJg17wp9Cn+TiYfMYNuFsx5YCJNhDmw3BcjH6ynRV8GjqW1Lz0+HtRDy1iTmGopa0xZ3RDjtNGBs73Mo7dXP64ZsK8y7IhJCfs4Cv1mExyDtnEf3E0Q1a4t1E8VkVNXOKuaoYjOudZ7kWQ4H+wVlxQDuQyA88WgC9Wj9oJjI2QvzNpDKvmehydF8kDpM1VXhUtPHIvuXd9jHvFG9bRQif17NxAFH8bvMdHnS7dCYQechS4yAuwpn6F/Fpkb9nAHIX8j1d53nb9h63q7U8wba10EHAMU9S2E9xLzocj4QNG4WBdZmiXZiqvyDMHUknW+dsDV/xzkQRqVmFClKmfUOfHk4aEqujLQd1Dvj5HrXMEw29XaQobs4o2I0Z+3ye0Iv4oQCeOFo2RYJoj4qKxPABsdgLJTVrhopNjU6VlYSUhKyBEFgMzsp75lwqv4Gyx8MPF/PZS0BPw7Wy49XW50MjOUEXvXv4synJpX3i0Xv4X5mTf+AUNNxA27aEmI0HvTU82PHCXwNUcosL36scgVvwd/2sv4mgDvC69J+qd4Jc7IrgercxHxBS9KvEcHc6vyS7FNh1awBFgiTb91EBrETvvsWsfFYg0mxQLzQ8lDVlaYr7i6hQ+YH6wiCQahJ7fPKamvjCo941HUqAYa/tojNhvH/CbT+eiUggx6JqOGL+FGq8Ko84LUVOjedfTXQUm7XjTOeZDSskZBASdTG5vrfvckQpLGjNUmPJtDHkJ5pksTU61QgzV2sTzD95xlWsZ0ZQldA1Wwg3dH3w9VZQbryWVuNCO3r3CVHH1/cAsowoYlCSv7KjYjQjMckTngCA1lWzZjLK71syjMA2uJQq68O81UOuX4yhzP+Iwctsj9FmsYLKUwO/JmjkMZdX9wHFhzGv00reIiIruZI/PbVJSrCVsRK4ROGXIhPONEWY5UYRAq2wIhwRxZN4UG5xQRPTg5/uUD42UmoioFlUz27omqvHmeY/O2H91WJfwBAZVCT1m7MiFezF5gXVJGrQLbNywngd+qUTkbWGRUi7ieFDRM0jHTKx40mdlpnSAJiWDgowOQUt9z0aYSr94DzQPOQqvUg3MsmQIuoIm39OPCtftyb5WJDHx9ExS3MgOX96CvV326fVMZMrvPtIYlXlhraE7ze8DXxeMd0A1JKmGLEkvAF/Ikq6hslLjZK7672zJy7OPVyu7r4ea3+InKOSWi9t4yXgFj3pHZDQ9BLF5im9I5pEQWeE21eZwJfP8Rc3eBlE33CJp1iywRhgHu/8yp7fcjDiIdWEfMMKbsSLT3iByn2+g/QiBTYm2Kn+0Hs9cnIMAJcmTc9SaDue5cUlb9DKY673Qc4yLOxnMXl8/R7H79/l4h7SeP/jZDBmbGRzO/TmR+qHwPSvdouP4Rg0kXBnsSCOpskuj3+WA2mVJKRBeacoKUVQacDqhNKMJ17be25F+3a7vZHQBvaSTHPqkvcsjkE+HaM3pNzF4vRdqE5dTZHhbVgBs0B0uiFXpEB8/Vk1URh06Q5OvAcw/l80nzuTouLpVScbF4Ff21VHE4Fh0oYp9bX/YZL1910UKdpCyhC036BYkDFIQH6oqTxqGw8giYC3Q8n+nokZcrCW6xNmABUsLDrDF3fEQHejIYPKdmyKbg3yb/dHOKgoZ57E2ejqEtMgWuGACtpdMFL2p51uyLzVoIqPv8Ac6V0pvh4dXQevjw+iu7WkWDzaaVnQ78osFQh9IbB+fFMfnh0PA4DeZmTvBG5gAL71QlkhYMlzWiP9De93lbxRi93k3Zgdm4u642iT3Q6+3YL/4tzBaYbGbhy59eAMSAck6AeL1PD7bVcvTjpwc/vaDj9FPjNL044MdaLNp+UVa38OgyC2+/4O5PJ0/zHVYEhFmPtvFzOOBw8MJqPZ0cjsf5Dh4UqzidHh1CswCsseeiJvMUmOLuaxUOq/UX+ehHaA/6XBgV0wlCzdAn910Yhl9pmb7wuxFwwCTIy2gqP3yNN6svYtTx+PuvoOB+uUG1fUQ5KlP4PropgvzriwNG35jRetKceJuJyC15cQCtX+Q/nZOfb2rZyAWFEGXfHwWw7BcHuTkIfNWamop+3eUFTMQqiwVsjX4eYIc0YI9qosFNdQwPOhlbPuvvHpcMs67BcLC/XE/dp8uvepQYBwUkCLPuykxCealLMw0EptgLyzQtMkqZsImQ7IFRp+jn45O/XJyeX/jnH9++Pf7w79Bh7yiDRiUjlWGJGe114AZIF6WhrGzpHherLV7ZO6M3qkoCfXOBsPxANHDs0YgqfwwtSr/kGlowQgA6okflIjFW0So0pOI/eyGLKN1+4H8++60u1iig3wWYf0RpP1zhcbkDEucC23V3e7ENgzs6lZvsKhohJeJ1MpGDjhIVi/ptozt60/2zEUqkP9IbmUkZobGi1pXou57B0dPxWKa9Qk9SIhkW/YPQSmegXWQDbSxAbz0WWXI3GSj4WRovVKI1lcHGTi4v+3N85G6uwrhwuGBoKX62LdqBXuFnVx2BOnF5msDI9AcsJdB1WVrk3+YqtxsL9RtdMTMy95NsIZwCi3wLiyFeUhCqA4VFTko9SEzS50WNZJsqGqiCp2g2mg3rLHJseDhRpQaGde64rFdGOOJ5GgIQrlMm9fGEMr/rO4Xm3fQ+W4VCryVY7PgzD9JCISCaoo/fUdDSc/2XEnpurqmQLaFL5EhrNtVLFhO7Ov748vWFf/zupX/85o1/fvYGv52fn364eP3+3bl/hp9fdlVbawZ/xRbQgHR6fLr5BcjzktEpxPXl4wcqyyt066hbPuIH8krr5OPLY7sNUwD4NwmAiwUj7eBXNy7rCCMVRQKFDiFZv5x9pCJ6FDm25dZiZV8XK7o6dSoGvJ6D5oWa+b73POgmwFwTv6uJUR0PtGWsKVutMQ5eOo/rrelFXtskrY02IBrN3YPWfZCeycEAndPUvQShtQal5AooFDdxX3McqwJzwOXag1jC9R69YJCe9nLrRSf+6mibOtBugEAz9esnT+kHafSbHVTBjvvonYh9+NqdNDzf9EZ7SNmHqFlWHZwDV5TRo6C1077lNrua810LYArmjbjuwaRGbGoH5xcfXp9c+GcfTi8+HL9+V3PFN++PX9Jpg5PRWV6ifQJVgrMvV05dGtKL7IrrfkhyvHQuacbqJBJD1AiNG/nkTnO0WxMExpjM6W+nJx+RT/hwyBDxfii9g/OFjI7Z1fXkgxJYCf76pri/oqjBoJ7bjvZaSYSeelv46xF1va1uyPXtQzkCPXGPw2DTeRErKdRl56Sob4ijm5oc6iJqzgJINB1aHYVIlLAZ1KLvG0uQcDpIEedVA7YqDj4U1cQb76WSpX5m2e4VYVF6HRdZivqLFGCcFiBn7KlPQyu/rdYg0MF2ckXBZEE3dLh5fX1ZSxnR0yZFO4h7pzUYimrpHtiaxmOkL4/hiadEmTq8Vb5lCvX+OOU2wFF2hTfT4GG9DgGT6qnGzD3i+nZZ7+B8r2jP8LN5Y3xvz8DjuyajcdfmtDbRJitu/VV82TNCXmQ5MFLMPBkPXLofKDoBhRw+fHg0/paVVEdO1at5F91YeODQcS/sVhlWfc4E/Kg+p5eoeRAzthZBUWCxicuouoki1ZcUKdP5DkDoR10oZsV+Yy4R7tnnwmHO4aAR1XpoBKFQ2XkufmIKK6o/F5Fn4bPnomvPLbxsGGN9PLzGSTmQ1uS5dfHq6PB5XWcn0KJvB3gz8yApDkCJtLjs+4Es+e7qyXkNxUtXIhpXwtGp2q03ol/OdI5qBZgb4mbYLW2aDtT2tU6j0G3/LcxaT80uI59vVTb5O81sNsVfretm3S1AdOlZgdRvVxp3TEeHY13tEDDnfdd0cVUxlsXKhM9+u8Z6OTSq5Lbqm6z7+biu+/lk0LxxTdL27cc3F69R0vqv3706/XD67uTUx4rIF/8u9YZOHV3cC/Na9/8bBNK/vCAN9izvtXY33pDUs+nTx3v6SYXo7fu/nPqkDIEORNPQiwe0iI7LCAiSxmqZ6tQPehbAGOivx2++YRg6Mk8f90GWumN669T30Fl/5FUHnRGAsNp43yoKfZiDyvgSVEe0yvinFsRPYrQRl1jPpk/mPZi3RHjNOAwJnuMUQ2H7S1NNlV8CwhgaFWDEcxi7x0hEj80m8kX9ceXHpYfoX0bHn7wExPxb6EpptKsoXdCDg0lsED8gn6cqAh4XK8BShuKnMeRbWhb5Gh80ik0o/qmh5pKXozTYES5WaLortB7f5q3QnSYIFow4qR+xC0JTl7Ce0pbuDubbqjxgnfGAK7FRkV/bqHfDibACaEUFrev33baSaM2lXBs20nOtfkBPP9WgwwhF67dAgiG2IzrgA1yfhqTpVA2mzXIFSvUwQJNBrIPHvvP2Sfpwev7x7Sn99Apmx9qKqsgXPVHPiYbsecszK3+wi05Oy07nA2VQRb8aTCN59LcQCV6TQfeo8zUBo+ECGmOU0209vdJDzVZQxPCCC8VtZexJ3yByFZvanrnkgzvV04FaN3JjaUpvewu4xgK7vsx6DHTDiBWUZTzXBRfyH1GZab8vrEcRmjUFV9CUWIR4gHg79XC6KiBo69X7N+Qni5mS7ANbn89Q/8UMQguvPDK/FA0rilJTDSPxAZPzuvk3FfPXSZVqtk6bvw5KNd/Fz6EoVnlfuSdN2LpoxR1FujR+2yLlJu+9H+neNVgtCDTG37wMR2fVrO5BTVukKJeS9k/Ovv61JX7TKvehdqLGwabS2RKC1NbteYd2e1lEwVW73kzDTW+NdDf+T8JXj6iJOEEH5Oa8ldp9112hdpeWtkC/OXi5DYkjYG5aFBpqAwlqeRJ4k8w1HnTq53SOLl6/PfV//vjyl9ML/8Pp8cmvpy99+OvkL2fvX7+78M+P/9rvj+5SOdThH/fGP1QT1GTvr312HRBdH22UeLkn8P6z1yATTEhhnQRAv1BUKM85pfErKpY/BNb8VRI08QSgLhrq5iDXyXDvYb+jRuQfYgX6gae/9x76xq+SGPNpnNRHnvydh0YrxVmZghvXLemeah+xCwT55z+o1pVXJYaqzA9hIXvXiubb6tHeZWNl9NdSiekfoV6/joXBhFTiLV4HN6rGXca88gciCF9URyXGgAka/Is2Qf+v1yw5h0P/uSS8D4ux90+fWsW/WkwJqHiFt+hUll/rLm0XS5Su7LP3J7+aoXrs38NnehRpbfE7tWntp0v7tWoNSLdqbfzW6D47EFWcO1lJnoua5aJi4Xy2rw55R6nx+jP+5Ib8Zvy2xrxVTiLPZ+aNI0+b1nN6LdRyT2hUel905ssbpbz5Sw6jflnGjybT8WH4tfMqa1cyURcptaGJVCNdxj18iI2ap0v84g/f0THLlIu6k3OjXrn8hUBMo+5pbt40bg6om9+9o14H5qjGbwz2DI199g8t5iqMfqXCdjgElFbLqcJ6j/b9E5MlKVPsDi5n3YNVDgadFlvfpl8Kd4TY8GaIr0PJFT+PiQFB+SsbdxZM7v3TCA+RmGkaEY1qt7Wwrv0lpmT6RzpNuljqHu2SEUFDRTPztFi5wcz1n3dVGUP/tY0wF3TQnzsubqe2NNrSV+88zbJrbXB9vzWo/G218PQrUm6a3TjVZ8/+eHFio3yh3081f15AROD2LcWgEVcmByr+OB5mcqgUtJP3b8/enF6c2h05ZqD5+RQX8n0y230fM85831blV/VsEc5Gk3HnRZRX1in9A+fNDDXwTyG7hBdWM9apg0pwfkoffP1PjCWCJQ=='))
assert hashlib.sha256(payload).hexdigest() == '60e783dc9e9308c1cf93d8ab8c8b9aed59e75041995f9edc673610ee201cd313'
for name, content in json.loads(payload).items():
    assert name in ['models.py', 'train_tw.py']
    (ROOT/name).write_text(content)
    compile(content, name, 'exec')

weight_hash = '0799f7efc93712944624417d2054a922e36849d5c57df754ddae559b8bd6ff0e'
candidates = list(Path('/kaggle/input').rglob('*ENSEMBLE_final.pt'))
matching = [p for p in candidates if hashlib.sha256(p.read_bytes()).hexdigest() == weight_hash]
assert matching, 'Attach notebook output: aiaila101/jpx-ensemble-norelu-top3-gap90 (version 1)'
WEIGHTS = ROOT/'JPX_ENSEMBLE_final.pt'
shutil.copy2(matching[0], WEIGHTS)
print('PRETRAINED_SOURCE', matching[0], 'SHA256', weight_hash, flush=True)

# These public repository files have the same hash as the user's supplied parts.
base = 'https://raw.githubusercontent.com/Kevin-Shih-Kai-Jen/jpx-equity-prediction/a79c2b2efe9a8472f52e61c94f062425e06a0c35/Taiwan_stock_data/dataset/'
part_hashes = ['d0d566ac77a3adea2ecb76b67a5c1d8ab70f654d6fcd9d4c9fdb56ba9b07db60',
               '11ae5c4c9fd3fc9680e078425eada246d84f1744856b57563a11bc359bf0e919',
               '6bcb9f3dd9ef9c02ee76f42d7f1b8d742f167f26edecb39028efbe0a3c23ec43']
DATA = ROOT/'stock_price_final.zip'
combined_hash = '1669cff3363ee845eaab96fb2482e909719e8b042f622debc5d6ea900bbcc036'
if not DATA.is_file() or hashlib.sha256(DATA.read_bytes()).hexdigest() != combined_hash:
    with DATA.open('wb') as target:
        for i, expected in enumerate(part_hashes, 1):
            url = base + f'stock_price_final.zip.{i:03d}'
            with urllib.request.urlopen(url, timeout=180) as response:
                part = response.read()
            assert hashlib.sha256(part).hexdigest() == expected, f'Part {i} hash mismatch'
            target.write(part)
            print('VERIFIED_DATA_PART', i, len(part), flush=True)
assert hashlib.sha256(DATA.read_bytes()).hexdigest() == combined_hash
print('COMPLETE_DATA_HASH_PASS', combined_hash, flush=True)
command = [sys.executable, '-u', str(ROOT/'train_tw.py'), '--data', str(DATA), '--weights', str(WEIGHTS)]
subprocess.run(command + ['--smoke-test', '--output', str(ROOT/'gpu_smoke_test')], check=True)
print('GPU_SMOKE_TEST_COMPLETE', flush=True)


In [ ]:
# Full expanding-window continuation: 40 folds, 2017 start, purged labels
with (RESULTS/'training_console.log').open('a', buffering=1) as log:
    process = subprocess.Popen(command + ['--output', str(RESULTS), '--max-seconds', '36000'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
        log.write(line)
    assert process.wait() == 0, 'Training failed; see training_console.log and resume_latest.pt'


In [ ]:
# Two requested conclusions, actual generated predictions only
import pandas as pd
from IPython.display import display, Image
for name in ['conclusion_all_validation.json', 'conclusion_from_2021_H2.json']:
    if (RESULTS/name).exists():
        print(name)
        print((RESULTS/name).read_text())
if (RESULTS/'backtest_metrics.csv').exists():
    display(pd.read_csv(RESULTS/'backtest_metrics.csv'))
    display(Image(filename=str(RESULTS/'backtest.png')))
archive = shutil.make_archive('/kaggle/working/TW_ENSEMBLE_Backtest_Results', 'zip', RESULTS)
print('RESULT_ARCHIVE', archive)
print('ALL_40_FOLDS_COMPLETE', (RESULTS/'completed.json').exists())
